# 01 · 승계 → 검증 → 인벤토리

**인벤토리를 새로 만들지 않는다.** 기존 저장소의 중복·출처·검수 판정을 승계하고,
그 위에 Head A 축을 파생한다 (D-06).

**이미지를 열지 않는다.** manifest 조인만 한다. 이미지 I/O는 02(crop)·03(품질) 소관.

## 이 노트북이 만드는 것 (D-12)

`excluded_by_policy` · `train_eligible_head_a` · `train_auxiliary` · `sample_role` ·
`eval_candidate_head_a` · `representative_rank` · 계보 플래그 3종

## 만들지 않는 것

`split` · `eval_eligible_head_a` · `canonical_eval_head_a` → **전부 노트북 04**

승계한 `split`은 폐기 대상인 기존 80/10/10 배정이므로 `legacy_split`으로 개명해
보존만 하고 어떤 파생에도 쓰지 않는다.

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "src"))

import pandas as pd

pd.set_option("display.max_rows", 120)
pd.set_option("display.width", 160)
print("repo root:", REPO_ROOT)

repo root: C:\Users\SSAFY\Documents\ssafy_jc\sub_pjt\banggoot_model


## 1. 승계 스냅샷

15개 대상을 `artifacts/inherited/`로 복사하고 SHA-256을 `MANIFEST.json`에 기록한다.
원본이 이전 승계 이후 바뀌었으면 **예외를 던진다** — 조용히 덮어쓰지 않는다.

특히 확인할 것:

- `dataset_inventory` — `label_path`가 여기에만 있다 (노트북 02의 crop 생성 전제)
- `origin_decisions` — origin 권위 소스. 없으면 강등 이전 값을 상속한다 (D-12)

In [2]:
from banggoot import inherit

manifest = inherit.snapshot()
print(f"승계 {manifest['entry_count']}건 · {manifest['generated_at']}")

pd.DataFrame(manifest["entries"])[["key", "kind", "bytes", "source_sha256"]].assign(
    source_sha256=lambda d: d.source_sha256.str[:12]
)

승계 17건 · 2026-08-03T06:47:57.801578+00:00


,key,kind,bytes,source_sha256
0,aihub_label_audit,file,36019,d4b6a7f3630d
1,aihub_label_audit_marker,file,1028,5bc6b98ef054
2,aihub_polygon_annotations,file,1545586,b9e78123295c
3,aihub_polygons,dir,5733406,658ae769cb21
4,artifact_decisions,file,2008051,8c96f68f3f34
5,dataset_inventory,file,18605716,d8059cf338c3
6,derivative_eval_review,file,4251,d2b17e15b22a
7,duplicate_groups,file,11632999,22969aa3bd12
8,eval_artifact_direct_decisions,file,168854,b88d7d01fb09
9,image_hashes,file,19016545,812e2c6f8ec2


In [3]:
checks = inherit.verify(manifest)
display(pd.DataFrame(checks))

bad = [c for c in checks if c["status"] != "ok"]
print(f"원본  ok {sum(c['source']   == 'ok' for c in checks)}/{len(checks)}")
print(f"복사본 ok {sum(c['snapshot'] == 'ok' for c in checks)}/{len(checks)}")
assert not bad, f"승계 무결성 실패: {bad}"

,key,kind,sha256,source,snapshot,status
0,aihub_label_audit,file,d4b6a7f3630d,ok,ok,ok
1,aihub_label_audit_marker,file,5bc6b98ef054,ok,ok,ok
2,aihub_polygon_annotations,file,b9e78123295c,ok,ok,ok
3,aihub_polygons,dir,658ae769cb21,ok,ok,ok
4,artifact_decisions,file,8c96f68f3f34,ok,ok,ok
5,dataset_inventory,file,d8059cf338c3,ok,ok,ok
6,derivative_eval_review,file,d2b17e15b22a,ok,ok,ok
7,duplicate_groups,file,22969aa3bd12,ok,ok,ok
8,eval_artifact_direct_decisions,file,b88d7d01fb09,ok,ok,ok
9,image_hashes,file,812e2c6f8ec2,ok,ok,ok


원본  ok 17/17
복사본 ok 17/17


## 2. 3-way 조인

```
dataset_inventory ⟕ split_manifest ⟕ origin_decisions   on record_id
```

세 파일 모두 54,797행이고 `record_id` 집합이 동일해야 한다.

In [4]:
from banggoot import metadata

joined = metadata.load_joined()
print("조인 결과:", joined.shape)
print("\n컬럼:")
print(sorted(joined.columns.tolist()))

조인 결과: (54797, 26)

컬럼:
['baseline_origin', 'candidate_target_class', 'dataset', 'duplicate_group_id', 'effective_origin', 'group_key', 'image_path', 'is_normal', 'label_path', 'legacy_split', 'license', 'mask_path', 'notes', 'origin_note', 'quarantine_kind', 'quarantined', 'record_id', 'reserved_reason', 'severity_grade', 'source_class', 'source_root_id', 'source_split', 'split_component_id', 'target_class', 'task', 'use_status']


### origin 2층 구조 확인 (D-12)

`split_manifest.source_origin`은 오버라이드가 반영되지 않은 baseline이다.
`roboflow_wall_defects`가 `real_verified`로 남아 있으면 **강등 이전 상태를 상속한 것**이다.

In [5]:
pd.crosstab(joined["dataset"], joined["effective_origin"])

effective_origin,dacon_derivative_probable,dacon_derivative_verified,real_verified,synthetic_verified,unknown_origin
dataset,,,,,
aihub_567,0,0,2728,0,0
dacon_wallpaper,0,0,4249,0,0
kaggle_cracks,0,0,40000,0,0
mvtec_ad,0,0,673,0,0
roboflow_house_defect,475,2228,0,976,2349
roboflow_wall_defects,0,0,0,1,375
roboflow_wallpaper_kr,14,729,0,0,0


### `reserved_inference` 컬럼 혼동 (D-08)

같은 832행이 두 컬럼에서 다른 이름을 갖는다.
`reserved_reason == 'reserved_inference'` 필터는 **오류 없이 0건**을 반환한다.

In [6]:
print("legacy_split == 'reserved_inference'          :",
      (joined.legacy_split == "reserved_inference").sum())
print("reserved_reason == 'contains_dacon_public_test':",
      (joined.reserved_reason == "contains_dacon_public_test").sum())
print("reserved_reason == 'reserved_inference'        :",
      (joined.reserved_reason == "reserved_inference").sum(), "  <- 함정")
print()
print(joined.reserved_reason.value_counts().to_string())

legacy_split == 'reserved_inference'          : 832
reserved_reason == 'contains_dacon_public_test': 832
reserved_reason == 'reserved_inference'        : 0   <- 함정

reserved_reason
                                        48198
no_eval_eligible_origin_in_component     3814
aihub_category_needs_review               945
contains_dacon_public_test                832
mvtec_official_protocol                   673
out_of_scope_category                     220
rare_detail_class_train_only              115


## 3. metadata v0 빌드

라벨 매핑 → eligibility 파생 → 계보 플래그 → 검증. 실패는 `AssertionError`로 던진다.

In [7]:
df = metadata.build()
print("metadata v0:", df.shape)
print()
print(df.sample_role.value_counts().to_string())

metadata v0: (54797, 46)

sample_role
excluded     44987
primary       6494
auxiliary     3316


### 공식 baseline 후보 vs ablation pool

**`head_a6_eligible` 은 `sample_role=='primary'` 만 포함한다.**
`train_eligible_head_a` 로 정의하면 auxiliary 3,169장(RHD 2,217 · RWKR 741 · RWD 211)이
baseline 에 새어 들어간다. auxiliary 는 ablation 전용이다 (D-12).


In [8]:
print(f"공식 baseline  head_a6_eligible  : {int(df.head_a6_eligible.sum()):>6,}")
print(f"ablation pool  head_a6_aux_pool   : {int(df.head_a6_aux_pool.sum()):>6,}")
print(f"탐색 7-class   head_a7_aux_train  : {int(df.head_a7_aux_train.sum()):>6,}")

assert df.loc[df.head_a6_eligible, "sample_role"].eq("primary").all()
assert not df.loc[df.head_a6_eligible, "train_auxiliary"].any()
assert not (df.head_a6_eligible & df.head_a6_aux_pool).any()

display(pd.crosstab(df[df.head_a6_eligible].unified_label,
                    df[df.head_a6_eligible].dataset, margins=True))

공식 baseline  head_a6_eligible  :  6,494
ablation pool  head_a6_aux_pool   :  3,169
탐색 7-class   head_a7_aux_train  :    103


dataset,aihub_567,dacon_wallpaper,kaggle_cracks,All
unified_label,,,,
breakage,793,1817,0,2610
crack,753,0,1500,2253
finish_damage,0,799,0,799
lifting,0,76,0,76
mold,0,144,0,144
stain_corrosion,0,612,0,612
All,1546,3448,1500,6494


### 평가 후보 × 출처 — PLAN §3.2 확정

**아직 split이 적용되지 않았다.** 이 표의 수치는 평가 표본이 아니라 *후보*다.
노트북 03(품질)·04(split·대표본)를 거치며 줄어든다.

출처가 2개 이상인 클래스만 leave-one-source-out이 가능하다.

In [9]:
e = df[df.eval_candidate_head_a & df.head_a6_eligible]
ct = pd.crosstab(e.unified_label, e.dataset)
display(ct)

print("\n클래스별 독립 출처 수 (LOSO 가능 여부):")
for label, row in ct.iterrows():
    srcs = [c for c in ct.columns if row[c] > 0]
    mark = "LOSO 가능" if len(srcs) >= 2 else "단일 출처 — 교차검증 불가"
    print(f"  {label:18s} {row.sum():6d}  출처 {len(srcs)}개 {srcs}  {mark}")

dataset,aihub_567,dacon_wallpaper,kaggle_cracks
unified_label,,,
breakage,793,1817,0
crack,753,0,1500
finish_damage,0,742,0
lifting,0,54,0
mold,0,144,0
stain_corrosion,0,595,0



클래스별 독립 출처 수 (LOSO 가능 여부):
  breakage             2610  출처 2개 ['aihub_567', 'dacon_wallpaper']  LOSO 가능
  crack                2253  출처 2개 ['aihub_567', 'kaggle_cracks']  LOSO 가능
  finish_damage         742  출처 1개 ['dacon_wallpaper']  단일 출처 — 교차검증 불가
  lifting                54  출처 1개 ['dacon_wallpaper']  단일 출처 — 교차검증 불가
  mold                  144  출처 1개 ['dacon_wallpaper']  단일 출처 — 교차검증 불가
  stain_corrosion       595  출처 1개 ['dacon_wallpaper']  단일 출처 — 교차검증 불가


## 4. 계보 배분과 봉인 (D-14)

`moisture_leak`은 평가 가능 이미지가 0장이라 공식 6-class 계보에서 제외된다.

**봉인은 행이 아니라 `split_component_id` 전체에 적용한다.** moisture 행만 봉인하면
같은 component의 `crack` 행이 aux 학습에 들어가 미래 holdout이 오염된다.

In [10]:
m = df[(df.dataset == "roboflow_wall_defects") & (df.unified_label == "moisture_leak")]
print(f"RWD moisture 총 {len(m)}")
print(f"  봉인 sealed_future_eval : {int(df.sealed_future_eval.sum())}행 "
      f"/ {df.loc[df.sealed_future_eval, 'split_component_id'].nunique()} component")
print(f"  탐색 head_a7_aux_train  : {int(df.head_a7_aux_train.sum())}")
print(f"  격리 quarantined        : {int(m.quarantined.sum())}")
print(f"  세 집합의 합집합        : {int((m.sealed_future_eval | m.head_a7_aux_train | m.quarantined).sum())} (= {len(m)} 이어야 함)")

RWD moisture 총 150
  봉인 sealed_future_eval : 45행 / 45 component
  탐색 head_a7_aux_train  : 103
  격리 quarantined        : 3
  세 집합의 합집합        : 150 (= 150 이어야 함)


### 봉인이 component 전체에 적용되는지 확인

실측상 `comp_040956`(crack×3 + moisture×1)과 `comp_040979`(crack×1 + moisture×2)가
혼합 component다. 이 둘이 봉인 대상으로 뽑히면 crack 행도 함께 봉인돼야 한다.

현재 seed에서 뽑히지 않았더라도 **메커니즘이 component 기준인지**를 직접 확인한다.

In [11]:
mixed = ["comp_040956", "comp_040979"]
display(df[df.split_component_id.isin(mixed)]
        .groupby(["split_component_id", "unified_label", "sealed_future_eval"]).size())

# 강제로 봉인시켜 crack 이 따라오는지 확인
forced = df.split_component_id.isin(mixed)
print(f"\n혼합 component 강제 봉인 시 대상 {int(forced.sum())}행 "
      f"(moisture 아닌 행 {int((forced & df.unified_label.ne('moisture_leak')).sum())})")
print(df[forced].groupby("unified_label").size().to_string())

split_component_id  unified_label  sealed_future_eval
comp_040956         crack          False                 3
                    moisture_leak  False                 1
comp_040979         crack          False                 1
                    moisture_leak  False                 2
dtype: int64


혼합 component 강제 봉인 시 대상 7행 (moisture 아닌 행 4)
unified_label
crack            4
moisture_leak    3


## 5b. AIHub 라벨 감사 승계 (D-15)

`audit_manifest.csv` 의 `annotation_error=true` 15건은 기존 B 파이프라인이 제외했던
라벨 오류다. 플래그만 달면 학습과 **평가 양쪽에** 재유입된다.
평가 유입이 더 위험하다 — 틀린 정답으로 지표를 계산하게 된다.

검수자는 `agent_visual_review` / `human_verified=false` 이므로 "확정 오류"가 아니라
`quality_status='review'` 로 표시하고 삭제하지 않는다.


In [12]:
e = df[df.legacy_annotation_error]
print(f"annotation_error {len(e)}건 · {dict(e.unified_label.value_counts())}")
print(f"  train_eligible_head_a : {int(e.train_eligible_head_a.sum())}")
print(f"  eval_candidate_head_a : {int(e.eval_candidate_head_a.sum())}")
print(f"  head_a6_eligible      : {int(e.head_a6_eligible.sum())}")
print(f"  quality_status        : {sorted(set(e.quality_status))}")

assert len(e) == 15
assert not e[["train_eligible_head_a", "eval_candidate_head_a", "head_a6_eligible"]].any().any()
assert e.quality_status.eq("review").all()

annotation_error 15건 · {'crack': np.int64(10), 'breakage': np.int64(5)}
  train_eligible_head_a : 0
  eval_candidate_head_a : 0
  head_a6_eligible      : 0
  quality_status        : ['review']


## 5. 제외 사유

In [13]:
df[df.rejection_reason != ""].rejection_reason.value_counts().to_frame("행 수")

,행 수
rejection_reason,
no_l1_mapping,23490
kaggle_subsample_not_selected,18500
reserved_aihub_category_needs_review,945
reserved_contains_dacon_public_test,832
license_excluded_dataset,673
synthetic_verified,303
reserved_out_of_scope_category,220
legacy_annotation_error,15
quarantined_stock_watermark,4


## 6. 저장

`metadata.csv`는 저장 후 즉시 재로딩해 한글 라벨 round-trip을 확인한다 (P8).

**게이트** — 아래 셀이 통과하면 노트북 02(crop 생성)로 넘어간다.

In [14]:
out_csv = metadata.save(df)
out_md = metadata.save_report(df)
print("saved:", out_csv)
print("saved:", out_md)

# 노트북 01 이 만들면 안 되는 컬럼 (D-12)
for col in ("split", "eval_eligible_head_a", "canonical_eval_head_a"):
    assert col not in df.columns, f"{col} 은 노트북 04 소관이다"

print("\n노트북 01 통과. 다음: 02_build_crops")

saved: C:\Users\SSAFY\Documents\ssafy_jc\sub_pjt\banggoot_model\artifacts\metadata\metadata.csv
saved: C:\Users\SSAFY\Documents\ssafy_jc\sub_pjt\banggoot_model\artifacts\reports\01_class_source.md

노트북 01 통과. 다음: 02_build_crops
